In [ ]:
import os
import logging
logging.getLogger("pypdf").setLevel(logging.ERROR)
 
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
 
os.environ["GROQ_API_KEY"] =os.getenv("GROQ_API_KEY", "")  # API KEY 
DB_PATH = "./chroma_db"
 

In [11]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_PATH, embedding_function=embedding_model)
print(f"Vector store loaded. Total vectors: {vectorstore._collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store loaded. Total vectors: 6183


In [12]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",   # <-- new model, llama-3.1-8b-instant deprecated 
    temperature=0.2
)
 

In [13]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})
 

In [14]:
prompt = ChatPromptTemplate.from_template("""You are a helpful assistant answering questions based ONLY on the provided context.
If the answer is not in the context, say "I don't have enough information in my documents to answer this."
Do not make up information.
 
Context:
{context}
 
Question: {question}
 
Answer:""")
 
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
 

In [15]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
 

In [16]:
def ask(question):
    print(f"\nQUESTION: {question}")
    answer = rag_chain.invoke(question)
    print(f"ANSWER: {answer}")
    docs = retriever.invoke(question)
    print("Sources:")
    seen = set()
    for doc in docs:
        source = os.path.basename(doc.metadata.get('source', 'unknown'))
        page = doc.metadata.get('page_label', '?')
        key = f"{source} (page {page})"
        if key not in seen:
            print(f"  - {key}")
            seen.add(key)
    return answer

In [18]:
ask("What is the power bi, why used  this ?")
ask("What is the application of machine learning ?")
ask("ujwal")


QUESTION: What is the power bi, why used  this ?
ANSWER: I don't have enough information in my documents to answer this.
Sources:
  - Learning_SQL.pdf (page 127)
  - Learning_SQL.pdf (page 193)
  - Learning_SQL.pdf (page xi)
  - Learning_SQL.pdf (page 126)

QUESTION: What is the application of machine learning ?
ANSWER: **Application of machine learning**

- Simplifies and improves solutions that would otherwise need extensive hand‑tuning or long rule lists (e.g., spam filtering).  
- Adapts to changing conditions and can be inspected to reveal the features it uses.  
- Provides insights into complex problems and large datasets, enabling discovery of hidden patterns (data mining).  
- Helps humans learn by uncovering new correlations and trends that were not obvious before.
Sources:
  - ML Notes.pdf (page 1)
  - ML Notes.pdf (page 12)
  - ML Notes.pdf (page 19)
  - ML Notes.pdf (page 3)

QUESTION: ujwal
ANSWER: I don't have enough information in my documents to answer this.
Sources:
 

"I don't have enough information in my documents to answer this."